In [1]:
import os, re, io, zipfile, requests, pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score


In [2]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

In [3]:
DATA_PATH = "spam.csv"
URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip"


In [5]:
def load_dataset(path=DATA_PATH):
    if not os.path.exists(path):
        print("🎤 Downloading dataset...")
        r = requests.get(URL)
        with zipfile.ZipFile(io.BytesIO(r.content)) as z:
            with z.open("SMSSpamCollection") as f:
                lines = [l.decode("utf-8", errors='ignore').strip().split("\t") for l in f.readlines()]
            df = pd.DataFrame(lines, columns=["label", "text"])
            df.to_csv(path, index=False)
            print("✅Dataset saved as spam.csv")
    else:
        df = pd.read_csv(path)
        print("✅Dataset loaded — total samples:", len(df))
    return df

In [6]:
df = load_dataset()
print(df.head())


🎤 Downloading dataset...
✅Dataset saved as spam.csv
  label                                               text
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


In [8]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [9]:
df["text"] = df["text"].apply(clean_text)
df["label"] = df["label"].map({"ham": 0, "spam": 1})


In [10]:
MAX_VOCAB = 20000
MAX_LEN = 100
EMBED_DIM = 128

In [11]:
tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token="<OOV>")
tokenizer.fit_on_texts(df["text"])
sequences = tokenizer.texts_to_sequences(df["text"])
X = pad_sequences(sequences, maxlen=MAX_LEN)
y = df["label"].values


In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)


In [13]:
model = Sequential([
Embedding(input_dim=MAX_VOCAB, output_dim=EMBED_DIM, input_length=MAX_LEN),
GRU(128, return_sequences=False),
Dropout(0.3),
Dense(64, activation="relu"),
Dropout(0.3),
Dense(1, activation="sigmoid")
])


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [14]:
model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [15]:
callbacks = [EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)]


In [16]:
history = model.fit(
X_train, y_train,
validation_data=(X_test, y_test),
epochs=10,
batch_size=64,
callbacks=callbacks,
verbose=1
)


Epoch 1/10
75/75 ━━━━━━━━━━━━━━━━━━━━ 21s 242ms/step - accuracy: 0.9158 - loss: 0.2481 - val_accuracy: 0.9833 - val_loss: 0.0539
Epoch 2/10
75/75 ━━━━━━━━━━━━━━━━━━━━ 17s 229ms/step - accuracy: 0.9918 - loss: 0.0295 - val_accuracy: 0.9845 - val_loss: 0.0541
Epoch 3/10
75/75 ━━━━━━━━━━━━━━━━━━━━ 20s 225ms/step - accuracy: 0.9968 - loss: 0.0128 - val_accuracy: 0.9869 - val_loss: 0.0559
Epoch 4/10
75/75 ━━━━━━━━━━━━━━━━━━━━ 20s 261ms/step - accuracy: 0.9987 - loss: 0.0063 - val_accuracy: 0.9821 - val_loss: 0.0595


In [17]:
preds = (model.predict(X_test) > 0.5).astype(int).flatten()
print("\n🎤 Classification Report:")
print(classification_report(y_test, preds, target_names=["ham", "spam"]))
print("✅Accuracy:", accuracy_score(y_test, preds))


27/27 ━━━━━━━━━━━━━━━━━━━━ 2s 67ms/step

🎤 Classification Report:
              precision    recall  f1-score   support

         ham       0.99      0.99      0.99       725
        spam       0.95      0.92      0.94       112

    accuracy                           0.98       837
   macro avg       0.97      0.96      0.96       837
weighted avg       0.98      0.98      0.98       837

✅Accuracy: 0.983273596176822


In [19]:
model.save("gru_spam_model.h5")
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)
print("🎤 Model and tokenizer saved!")

🎤 Model and tokenizer saved!


In [21]:
def predict_spam(text):
    text = clean_text(text)
    seq = tokenizer.texts_to_sequences([text])
    X = pad_sequences(seq, maxlen=MAX_LEN)
    prob = float(model.predict(X)[0][0])
    label = "SPAM 🎤" if prob >= 0.5 else "HAM ✅"
    print(f"🎤 Text: {text[:100]}...")
    print(f"🎤 Prediction: {label} (prob={prob:.4f})\n")

In [22]:
samples = [
"Congratulations! You've won a free vacation. Click here to claim.",
"Are you coming to the meeting tomorrow morning?",
"You have been selected for a $1000 gift card. Reply WIN to claim now!",
"Don't forget to submit your project before 5 PM today."
]


In [24]:
for s in samples:
    predict_spam(s)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step
🎤 Text: congratulations you ve won a free vacation click here to claim...
🎤 Prediction: SPAM 🎤 (prob=0.9387)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step
🎤 Text: are you coming to the meeting tomorrow morning...
🎤 Prediction: HAM ✅ (prob=0.0012)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step
🎤 Text: you have been selected for a 1000 gift card reply win to claim now...
🎤 Prediction: SPAM 🎤 (prob=0.9983)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step
🎤 Text: don t forget to submit your project before 5 pm today...
🎤 Prediction: HAM ✅ (prob=0.0497)

